## Portfolio 2

In deze portfolio gaan we prompt engineering toepassen om de data van de HBO Kennisbank te webscrapen.

Daarna gaan we met deze data titelgeneratie toepassen. Dit gaan we verder finetunen, sampelen, RLAIF en evalueren. 

### Prompt 1

### Antwoord 1

### Resultaat verwerken

In [ ]:
# stap 2
import requests
import time
import pandas as pd
from bs4 import BeautifulSoup

url = "https://hbo-kennisbank.nl/searchresult?q="
headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(url, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")

titles = soup.find_all("h2")  # voorbeeld, moet je inspecteren

for t in titles:
    print(t.get_text(strip=True))

# stap 3
base_url = "https://hbo-kennisbank.nl/searchresult?page={}"

for page in range(1, 50):
    url = base_url.format(page)
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # parse titles

# stap 4
links = []

for item in soup.select("a.result-link"):
    links.append(item["href"])

# stap 5
def scrape_detail(url):
    r = requests.get(url, headers=headers)
    s = BeautifulSoup(r.text, "html.parser")

    title = s.find("h1").get_text(strip=True)
    abstract = s.find("div", class_="abstract")

    return {
        "title": title,
        "abstract": abstract.get_text(strip=True) if abstract else None
    }

# stap 6

data = []

for link in links:
    try:
        item = scrape_detail(link)
        data.append(item)
    except Exception as e:
        print("Error:", e)

# stap 7
time.sleep(1)

# stap 8

df = pd.DataFrame(data)
df.to_csv("hbo_kennisbank.csv", index=False)

### Prompt 2

### Antwoord 2

### Resultaat verwerken

Aanpassingen: 
- 'SEARCH_URL' lichtelijk aangepast zodat het scraper bij meerdere pagina's kan komen.
- 'max_pages' aangepast zodat het meer pagina's scrapet.

In [18]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm
from urllib.parse import urljoin

BASE_URL = "https://hbo-kennisbank.nl"
SEARCH_URL = BASE_URL + "/searchresult?has-link=yes&p={}"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"
}

# -----------------------------
# Helper: veilige tekst extractie
# -----------------------------
def safe_text(element):
    return element.get_text(strip=True) if element else None


# -----------------------------
# Stap 1: Verzamel links per pagina
# -----------------------------
def get_result_links(page):
    url = SEARCH_URL.format(page)
    response = requests.get(url, headers=HEADERS)
    
    if response.status_code != 200:
        print(f"Fout bij pagina {page}")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    links = []
    
    # Selector is bewust breed gehouden (site kan variëren)
    for a in soup.select("a"):
        href = a.get("href")
        if href and "/details/" in href:
            full_url = urljoin(BASE_URL, href)
            links.append(full_url)

    return list(set(links))  # duplicates verwijderen


# -----------------------------
# Stap 2: Scrape detailpagina
# -----------------------------
def scrape_detail(url):
    try:
        response = requests.get(url, headers=HEADERS)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, "html.parser")

        # Titel
        title = safe_text(soup.find("h1"))

        # Abstract (meerdere fallback opties)
        abstract = None
        
        # optie 1
        abstract_div = soup.find("div", class_="abstract")
        if abstract_div:
            abstract = safe_text(abstract_div)

        # optie 2 fallback
        if not abstract:
            for div in soup.find_all("div"):
                if "samenvatting" in div.get_text().lower():
                    abstract = safe_text(div)
                    break

        # Jaar (meerdere strategieën)
        year = None

        # optie 1: metadata blok
        for li in soup.find_all("li"):
            text = li.get_text().lower()
            if "jaar" in text or "datum" in text:
                year = li.get_text(strip=True)
                break

        # optie 2 fallback: zoek 4-digit jaar
        if not year:
            import re
            text = soup.get_text()
            match = re.search(r"\b(20\d{2}|19\d{2})\b", text)
            if match:
                year = match.group(0)

        return {
            "title": title,
            "abstract": abstract,
            "year": year,
            "url": url
        }

    except Exception as e:
        print(f"Error bij {url}: {e}")
        return None


# -----------------------------
# Stap 3: Hoofd scraper
# -----------------------------
def scrape_hbo_kennisbank(max_pages=10, delay=1):
    all_links = []

    print("Links verzamelen...")
    for page in range(1, max_pages + 1):
        links = get_result_links(page)
        print(f"Pagina {page}: {len(links)} links")
        all_links.extend(links)
        time.sleep(delay)

    all_links = list(set(all_links))
    print(f"Totaal unieke links: {len(all_links)}")

    # Detail scraping
    data = []
    print("Detailpagina's scrapen...")

    for link in tqdm(all_links):
        item = scrape_detail(link)
        if item and item["title"]:
            data.append(item)
        time.sleep(delay)

    df = pd.DataFrame(data)
    return df


# -----------------------------
# Run scraper
# -----------------------------
if __name__ == "__main__":
    df = scrape_hbo_kennisbank(max_pages=50, delay=1)

    print(df.head())

    df.to_csv("hbo_kennisbank_data.csv", index=False)
    print("Data opgeslagen als hbo_kennisbank_data.csv")

Links verzamelen...
Pagina 1: 10 links
Pagina 2: 10 links
Pagina 3: 9 links
Pagina 4: 9 links
Pagina 5: 6 links
Pagina 6: 9 links
Pagina 7: 6 links
Pagina 8: 4 links
Pagina 9: 10 links
Pagina 10: 7 links
Pagina 11: 5 links
Pagina 12: 2 links
Pagina 13: 4 links
Pagina 14: 7 links
Pagina 15: 8 links
Pagina 16: 8 links
Pagina 17: 10 links
Pagina 18: 8 links
Pagina 19: 10 links
Pagina 20: 10 links
Pagina 21: 10 links
Pagina 22: 2 links
Pagina 23: 10 links
Pagina 24: 8 links
Pagina 25: 8 links
Pagina 26: 7 links
Pagina 27: 2 links
Pagina 28: 1 links
Pagina 29: 1 links
Pagina 30: 2 links
Pagina 31: 10 links
Pagina 32: 1 links
Pagina 33: 3 links
Pagina 34: 6 links
Pagina 35: 8 links
Pagina 36: 8 links
Pagina 37: 6 links
Pagina 38: 5 links
Pagina 39: 8 links
Pagina 40: 7 links
Pagina 41: 3 links
Pagina 42: 10 links
Pagina 43: 10 links
Pagina 44: 10 links
Pagina 45: 9 links
Pagina 46: 8 links
Pagina 47: 8 links
Pagina 48: 8 links
Pagina 49: 9 links
Pagina 50: 7 links
Totaal unieke links: 343
De

100%|██████████| 343/343 [07:44<00:00,  1.35s/it]

                                               title  \
0  Het gebruik van een tablet in de klas; een vlo...   
1          A meaningful and vibrant work environment   
2     Jouw ervaringen en ervaringskennis: werkbladen   
3  Urban Upcycling: Van experiment naar haalbaar ...   
4                        TechWijs in Zorg en Welzijn   

                                            abstract  year  \
0  Het gebruik van een tablet in de klas; een vlo...  2019   
1  A meaningful and vibrant work environmentMars,...  2025   
2  Jouw ervaringen en ervaringskennis: werkbladen...  2026   
3  Urban Upcycling: Van experiment naar haalbaar ...  2050   
4  TechWijs in Zorg en WelzijnTechnologiecompeten...  None   

                                                 url  
0  https://hbo-kennisbank.nl/details/hanzepure:oa...  
1  https://hbo-kennisbank.nl/details/hanzepure:oa...  
2  https://hbo-kennisbank.nl/details/sharekit_win...  
3  https://hbo-kennisbank.nl/details/amsterdam_pu...  
4  https://hbo-

### Prompt 3

### Antwoord 3

### Verwerken

In [22]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
from tqdm import tqdm
from urllib.parse import urljoin

BASE_URL = "https://hbo-kennisbank.nl"
SEARCH_URL = BASE_URL + "/searchresult?has-link=yes&p={}"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7)"
}

# -----------------------------
# Helper: veilige tekst extractie
# -----------------------------
def safe_text(element):
    return element.get_text(strip=True) if element else None


# -----------------------------
# Stap 1: Verzamel links per pagina
# -----------------------------
def get_result_links(page):
    url = SEARCH_URL.format(page)
    response = requests.get(url, headers=HEADERS)
    
    if response.status_code != 200:
        print(f"Fout bij pagina {page}")
        return []

    soup = BeautifulSoup(response.text, "html.parser")

    links = []
    
    # Selector is bewust breed gehouden (site kan variëren)
    for a in soup.select("a"):
        href = a.get("href")
        if href and "/details/" in href:
            full_url = urljoin(BASE_URL, href)
            links.append(full_url)

    return list(set(links))  # duplicates verwijderen


# -----------------------------
# Stap 2: Scrape detailpagina
# -----------------------------
def scrape_detail(url):
    try:
        response = requests.get(url, headers=HEADERS)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, "html.parser")

        # Titel
        title = safe_text(soup.find("h1"))

        # Abstract (meerdere fallback opties)
        abstract = None
        
        # optie 1
        abstract_div = soup.find("div", class_="abstract")
        if abstract_div:
            abstract = safe_text(abstract_div)

        # optie 2 fallback
        if not abstract:
            for div in soup.find_all("div"):
                if "samenvatting" in div.get_text().lower():
                    abstract = safe_text(div)
                    break

        # Jaar (meerdere strategieën)
        year = None

        # optie 1: metadata blok
        for li in soup.find_all("li"):
            text = li.get_text().lower()
            if "jaar" in text or "datum" in text:
                year = li.get_text(strip=True)
                break

        # optie 2 fallback: zoek 4-digit jaar
        if not year:
            import re
            text = soup.get_text()
            match = re.search(r"\b(20\d{2}|19\d{2})\b", text)
            if match:
                year = match.group(0)
        
        # -----------------------------
        # Datum + categorie + type
        # -----------------------------
        date = None
        category = None
        pub_type = None

        meta = soup.find("p", class_="meta subtle")

        if meta:
            spans = meta.find_all("span")

            # Filter alleen echte tekst-spans (icon spans hebben vaak geen tekst)
            values = [s.get_text(strip=True) for s in spans if s.get_text(strip=True)]

            # Verwachte structuur:
            # [datum, categorie, publicatietype]
            if len(values) >= 1:
                date = values[0]
            if len(values) >= 2:
                category = values[1]
            if len(values) >= 3:
                pub_type = values[2]

        return {
            "title": title,
            "abstract": abstract,
            "year": year,
            "full_date": date,
            "category": category,
            "publication_type": pub_type,
            "url": url
        }

    except Exception as e:
        print(f"Error bij {url}: {e}")
        return None


# -----------------------------
# Stap 3: Hoofd scraper
# -----------------------------
def scrape_hbo_kennisbank(max_pages=10, delay=1):
    all_links = []

    print("Links verzamelen...")
    for page in range(1, max_pages + 1):
        links = get_result_links(page)
        print(f"Pagina {page}: {len(links)} links")
        all_links.extend(links)
        time.sleep(delay)

    all_links = list(set(all_links))
    print(f"Totaal unieke links: {len(all_links)}")

    # Detail scraping
    data = []
    print("Detailpagina's scrapen...")

    for link in tqdm(all_links):
        item = scrape_detail(link)
        if item and item["title"]:
            data.append(item)
        time.sleep(delay)

    df = pd.DataFrame(data)
    return df


# -----------------------------
# Run scraper
# -----------------------------
if __name__ == "__main__":
    df = scrape_hbo_kennisbank(max_pages=10, delay=1)

    print(df.head())

    df.to_csv("hbo_kennisbank_data.csv", index=False)
    print("Data opgeslagen als hbo_kennisbank_data.csv")

Links verzamelen...
Pagina 1: 10 links
Pagina 2: 10 links
Pagina 3: 9 links
Pagina 4: 9 links
Pagina 5: 6 links
Pagina 6: 9 links
Pagina 7: 5 links
Pagina 8: 4 links
Pagina 9: 9 links
Pagina 10: 7 links
Totaal unieke links: 78
Detailpagina's scrapen...


100%|██████████| 78/78 [01:31<00:00,  1.17s/it]

                                               title  \
0  Verpleegkundig perspectief op persoonsgerichte...   
1                  Op koers blijven als zijinstromer   
2     Jouw ervaringen en ervaringskennis: werkbladen   
3  Urban Upcycling: Van experiment naar haalbaar ...   
4        Ervaringskennis in de organisatie: werkblad   

                                            abstract  year full_date category  \
0  Verpleegkundig perspectief op persoonsgerichte...  2026      None     None   
1  Op koers blijven als zijinstromerkwaliteitsvol...  2009      None     None   
2  Jouw ervaringen en ervaringskennis: werkbladen...  2021      None     None   
3  Urban Upcycling: Van experiment naar haalbaar ...  2050      None     None   
4  Ervaringskennis in de organisatie: werkbladLie...  2021      None     None   

  publication_type                                                url  
0             None  https://hbo-kennisbank.nl/details/sharekit_win...  
1             None  https://hbo-

In [19]:
pd.read_csv("hbo_kennisbank_data.csv")

,title,abstract,year,url
0,Het gebruik van een tablet in de klas; een vlo...,Het gebruik van een tablet in de klas; een vlo...,2019.0,https://hbo-kennisbank.nl/details/hanzepure:oa...
1,A meaningful and vibrant work environment,"A meaningful and vibrant work environmentMars,...",2025.0,https://hbo-kennisbank.nl/details/hanzepure:oa...
2,Jouw ervaringen en ervaringskennis: werkbladen,Jouw ervaringen en ervaringskennis: werkbladen...,2026.0,https://hbo-kennisbank.nl/details/sharekit_win...
3,Urban Upcycling: Van experiment naar haalbaar ...,Urban Upcycling: Van experiment naar haalbaar ...,2050.0,https://hbo-kennisbank.nl/details/amsterdam_pu...
4,TechWijs in Zorg en Welzijn,TechWijs in Zorg en WelzijnTechnologiecompeten...,NaN,https://hbo-kennisbank.nl/details/saxion_kenni...
...,...,...,...,...
338,Meer dan fietsen voor het klimaat: naar effect...,Meer dan fietsen voor het klimaat: naar effect...,2024.0,https://hbo-kennisbank.nl/details/amsterdam_pu...
339,Versterking van kennisdeling en regionale same...,Versterking van kennisdeling en regionale same...,2021.0,https://hbo-kennisbank.nl/details/sharekit_han...
340,Academic stress and mental fatigue predict sub...,Academic stress and mental fatigue predict sub...,2026.0,https://hbo-kennisbank.nl/details/hanzepure:oa...
341,Biogas digestate in closed-loop digeponic syst...,Biogas digestate in closed-loop digeponic syst...,2024.0,https://hbo-kennisbank.nl/details/aereshogesch...
